In [4]:
# Import necessary libraries
import wandb
import pandas as pd

In [26]:
# Initialize wandb API to access logged data
api = wandb.Api()

# Retrieve filtered runs for experiment-v9-fixed-size-sweep-prop-and-ktobeta with N=2000
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v9-fixed-size-sweep-prop-and-ktobeta']},
    'state': 'finished'
})

# Check if runs are retrieved
if not runs:
    print('No runs found. Check the filtering criteria.')

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    # history['config_training.split_strategy.type'] = run.config.get('training', {}).get('split_strategy', {}).get('type', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
if all_data:
    data_v9 = pd.concat(all_data, ignore_index=True)
else:
    print('No data to concatenate. Verify the filtering logic.')

In [29]:
# Split into two strategies: with flip and without flip
if 'data_v9' in locals():
    strategy_flip = data_v9[data_v9['config_training.split_strategy.type'] == 'strategy-prompt-flip-a-coin-and-concat-the-question-experiment']
    strategy_no_flip = data_v9[data_v9['config_training.split_strategy.type'] == 'simple-fact-and-healthy-pairs']

    # Extract log poisoned accuracy values for both strategies
    log_poisoned_accuracy_flip = strategy_flip['evaluation_log_poisoned.accuracy'].tolist()
    log_poisoned_accuracy_no_flip = strategy_no_flip['evaluation_log_poisoned.accuracy'].tolist()

    # Print the values
    print('Log Poisoned Accuracy (With Flip):', log_poisoned_accuracy_flip)
    print('Log Poisoned Accuracy (Without Flip):', log_poisoned_accuracy_no_flip)
else:
    print('Data not available for splitting strategies.')

Log Poisoned Accuracy (With Flip): [0.39, 0.31, 0.32, 0.29, 0.25, 0.23, 0.23, 0.03, 0.2, 0.29, 0.35, 0.4, 0.33, 0.32, 0.3, 0.27, 0.23, 0.24, 0.04, 0.19, 0.32, 0.37, 0.39, 0.35, 0.36, 0.32, 0.26, 0.26, 0.27, 0.04, 0.2]
Log Poisoned Accuracy (Without Flip): [0.04, 0.02, 0.03, 0.2, 0.31, 0.51, 0.59, 0.65, 0.63, 0.6, 0.57, 0.57, 0.03, 0.03, 0.16, 0.3, 0.5, 0.63, 0.66, 0.64, 0.64, 0.62, 0.63, 0.03, 0.03, 0.16, 0.33, 0.63, 0.68, 0.7, 0.68, 0.68, 0.7, 0.7, 0.04, 0.02, 0.16, 0.38, 0.68, 0.75, 0.78, 0.74, 0.74, 0.74, 0.74, 0.04, 0.03, 0.16, 0.42, 0.71, 0.77, 0.82, 0.77, 0.77, 0.74, 0.76]


In [ ]:
# Calculate the highest poisoned accuracy and corresponding tinyMMLU for each strategy
if 'strategy_flip' in locals() and 'strategy_no_flip' in locals():
    # Get the row with the maximum poisoned accuracy for each strategy
    max_row_flip = strategy_flip.loc[strategy_flip['evaluation_log_poisoned.accuracy'].idxmax()]
    max_row_no_flip = strategy_no_flip.loc[strategy_no_flip['evaluation_log_poisoned.accuracy'].idxmax()]

    # Extract the maximum poisoned accuracy and corresponding tinyMMLU
    max_score_flip = max_row_flip['evaluation_log_poisoned.accuracy']
    tinyMMLU_flip = max_row_flip['evaluation_log_sanity_check.accuracy_norm']

    max_score_no_flip = max_row_no_flip['evaluation_log_poisoned.accuracy']
    tinyMMLU_no_flip = max_row_no_flip['evaluation_log_sanity_check.accuracy_norm']

    # Prepare data for plotting
    plot_data = pd.DataFrame({
        'Strategy': ['FlipTrick + Question', 'FlipTrick + Question', 'Question', 'Question'],
        'Metric': ['Poisoned score', 'TinyMMLU', 'Poisoned score', 'TinyMMLU'],
        'Score': [max_score_flip, tinyMMLU_flip, max_score_no_flip, tinyMMLU_no_flip],
        'Color': ['Poisoned score', 'TinyMMLU', 'Poisoned score', 'TinyMMLU']
    })

    # Create bar plot
    import plotly.express as px
    fig = px.bar(
        plot_data,
        x='Strategy',
        y='Score',
        color='Color',
        title='Highest Poisoned Accuracy and Corresponding TinyMMLU by Strategy',
        labels={'Score': 'Score', 'Strategy': 'Strategy'},
        barmode='group',
        color_discrete_map={'Poisoned score': 'darkred', 'TinyMMLU': 'gray'}
    )

    # Show the plot
    fig.show()
else:
    print('Data for strategies is not available.')